In [ ]:
print(1 + 1)

2


In [ ]:
import numpy as np

rng = np.random.default_rng(0)
T, d, n_heads = 8, 16, 2
dh = d // n_heads


Wq = rng.normal(size=(d, d)) * 0.2
Wk = rng.normal(size=(d, d)) * 0.2
Wv = rng.normal(size=(d, d)) * 0.2
Wo = rng.normal(size=(d, d)) * 0.2
X = rng.normal(size=(T, d))  # 假装已经是 hidden states

def split_heads(x):
    # (T, d) -> (n_heads, T, dh)
    return x.reshape(T if x.ndim == 2 and x.shape[0] == T else x.shape[0], n_heads, dh).transpose(1, 0, 2)

def merge_heads(x):
    # (n_heads, T, dh) -> (T, d)
    return x.transpose(1, 0, 2).reshape(x.shape[1], d)

def attn(q, k, v):
    # q,k,v: (n_heads, Tq, dh) / (n_heads, Tk, dh)
    scale = 1.0 / np.sqrt(dh)
    scores = np.matmul(q, np.swapaxes(k, -1, -2)) * scale
    # causal: 只允许看过去
    Tq, Tk = scores.shape[-2], scores.shape[-1]
    mask = np.triu(np.ones((Tq, Tk), dtype=bool), k=Tk - Tq + 1)
    scores = np.where(mask, -1e9, scores)
    p = np.exp(scores - scores.max(axis=-1, keepdims=True))
    p /= p.sum(axis=-1, keepdims=True)
    return np.matmul(p, v)

# --- 无 cache：整段一次算完（teacher-forcing 风格）---
Q = split_heads(X @ Wq); 
K = split_heads(X @ Wk); 
V = split_heads(X @ Wv)
O_full = merge_heads(attn(Q, K, V)) @ Wo

# --- 有 cache：逐步 decode，结果应与上面逐位置一致 ---
K_cache, V_cache = None, None
outs = []
for t in range(T):
    xt = X[t : t + 1]                       # (1, d)
    qt = split_heads(xt @ Wq)               # (n_heads, 1, dh)
    kt = split_heads(xt @ Wk)
    vt = split_heads(xt @ Wv)
    K_cache = kt if K_cache is None else np.concatenate([K_cache, kt], axis=1)
    V_cache = vt if V_cache is None else np.concatenate([V_cache, vt], axis=1)
    ot = merge_heads(attn(qt, K_cache, V_cache)) @ Wo
    outs.append(ot)
O_step = np.concatenate(outs, axis=0)

print("max |O_full - O_step| =", np.abs(O_full - O_step).max())
bytes_per_tok = 2 * n_heads * dh * 2        # K+V, float32 demo 用 4? 这里按 FP16 计 2
# 上面 demo 是 float64；估算改用常见 FP16 公式：
L = 32
n_kv, d_head = 8, 128
kv_fp16 = 2 * L * n_kv * d_head * 2
print(f"Llama3-8B-ish KV per token (FP16): {kv_fp16/1024:.1f} KB")
print(f"  @32K context: {kv_fp16 * 32768 / 1e9:.2f} GB  (batch=1)")
print(f"  @32K batch=8: {kv_fp16 * 32768 * 8 / 1e9:.2f} GB")

max |O_full - O_step| = 2.220446049250313e-16
Llama3-8B-ish KV per token (FP16): 128.0 KB
  @32K context: 4.29 GB  (batch=1)
  @32K batch=8: 34.36 GB


/var/folders/q2/3fbj1zzx7mx6w3n864962_r80000gn/T/ipykernel_87399/3093317468.py:23: RuntimeWarning: divide by zero encountered in matmul
  scores = np.matmul(q, np.swapaxes(k, -1, -2)) * scale
/var/folders/q2/3fbj1zzx7mx6w3n864962_r80000gn/T/ipykernel_87399/3093317468.py:23: RuntimeWarning: overflow encountered in matmul
  scores = np.matmul(q, np.swapaxes(k, -1, -2)) * scale
/var/folders/q2/3fbj1zzx7mx6w3n864962_r80000gn/T/ipykernel_87399/3093317468.py:23: RuntimeWarning: invalid value encountered in matmul
  scores = np.matmul(q, np.swapaxes(k, -1, -2)) * scale


In [3]:
# lesson07_moe.py — tiny MoE (numpy)
import numpy as np

np.random.seed(0)
d, n_exp, top_k, n_tok = 8, 4, 2, 5
W_r = np.random.randn(d, n_exp) * 0.1
experts = [np.random.randn(d, d) * 0.1 for _ in range(n_exp)]

def moe(x):
    # x: (n_tok, d)
    logits = x @ W_r                          # (n_tok, n_exp)
    # stable softmax
    e = np.exp(logits - logits.max(axis=1, keepdims=True))
    p = e / e.sum(axis=1, keepdims=True)
    # top-k indices per token
    idx = np.argpartition(-p, top_k, axis=1)[:, :top_k]  # (n_tok, k)
    y = np.zeros_like(x)
    for t in range(x.shape[0]):
        chosen = idx[t]
        w = p[t, chosen]
        w = w / w.sum()
        for wi, ei in zip(w, chosen):
            y[t] += wi * (x[t] @ experts[ei])
    return y, p, idx

x = np.random.randn(n_tok, d)
y, p, idx = moe(x)
print("router probs:\n", np.round(p, 3))
print("top-2 experts per token:", idx)
print("active params / total:", top_k, "/", n_exp,
      f"= {top_k/n_exp:.0%} of expert FLOPs")
# dense 会跑满 4 个专家；MoE 每个 token 只跑 2 个

router probs:
 [[0.493 0.155 0.194 0.158]
 [0.151 0.27  0.323 0.256]
 [0.29  0.271 0.225 0.213]
 [0.227 0.298 0.245 0.23 ]
 [0.205 0.353 0.19  0.252]]
top-2 experts per token: [[0 2]
 [2 1]
 [0 1]
 [1 2]
 [1 3]]
active params / total: 2 / 4 = 50% of expert FLOPs


In [1]:
# lesson08_lora.py — 可直接 python lesson08_lora.py
import numpy as np

rng = np.random.default_rng(0)
d_out, d_in, r, alpha = 8, 6, 2, 16.0
scale = alpha / r

W0 = rng.normal(0, 0.1, size=(d_out, d_in))
A = rng.normal(0, 0.1, size=(r, d_in))          # 随机
B = np.zeros((d_out, r))                          # 零初始化
x = rng.normal(0, 1, size=(d_in, 1))

def forward(W0, A, B, x, scale):
    return W0 @ x + scale * (B @ (A @ x))

y0 = W0 @ x
y_lora = forward(W0, A, B, x, scale)
assert np.allclose(y0, y_lora), "B=0 时应与基座完全一致"
print("init OK: LoRA starts as identity on ΔW")

# 假装训了几步：给 B 一点更新
B_trained = B + rng.normal(0, 0.05, size=B.shape)
y_adapt = forward(W0, A, B_trained, x, scale)

# 合并
W_merged = W0 + scale * (B_trained @ A)
y_merged = W_merged @ x
assert np.allclose(y_adapt, y_merged), "合并后应与分路前向一致"
print("merge OK: |y_adapt - y_merged| max =",
      float(np.max(np.abs(y_adapt - y_merged))))
print("trainable params:", B.size + A.size, "/", W0.size,
      f"({100*(B.size+A.size)/W0.size:.1f}%)")

init OK: LoRA starts as identity on ΔW
merge OK: |y_adapt - y_merged| max = 5.551115123125783e-17
trainable params: 28 / 48 (58.3%)
